# PD-AMM Convergence Demo

Shows how `InformedTrader` drives a K=1 Normal AMM toward a target distribution.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))

import math
import numpy as np
import matplotlib.pyplot as plt

from pdamm.amm.gaussian_mixture import GaussianMixtureAMM, GaussianMixtureParams
from pdamm.agents.informed_trader import InformedTrader

K_RUST = 5.0 * math.sqrt(10.0 * math.sqrt(math.pi))

def make_amm(mu=95.0, sigma=10.0, b=50.0):
    return GaussianMixtureAMM(
        b=b, k=K_RUST,
        params=GaussianMixtureParams.single(mu=mu, sigma=sigma),
        max_collateral_per_trade=50.0,
    )


## Single InformedTrader convergence

In [ ]:
amm = make_amm(mu=95.0)
posterior = GaussianMixtureParams.single(mu=110.0, sigma=8.0)
trader = InformedTrader("i0", posterior=posterior, step_fraction=0.25, max_fraction_of_b=0.15)

history = []
for step in range(100):
    trader.step(amm)
    history.append(amm.current_mu)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history, label="AMM mu")
ax.axhline(110.0, color="red", linestyle="--", label="True mu (posterior)")
ax.axhline(95.0, color="gray", linestyle=":", label="Initial mu")
ax.set_xlabel("Step")
ax.set_ylabel("Distribution mean (mu)")
ax.set_title("InformedTrader drives AMM toward posterior")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(os.path.dirname(os.path.abspath("__file__")),
    "../notebooks/convergence_demo.png"), dpi=120)
plt.show()
print(f"Final AMM mu: {amm.current_mu:.2f}  (target: 110.0)")
print(f"Trades executed: {trader.trades_executed}")


## Collateral paid vs. distribution shift

In [ ]:
amm2 = make_amm(mu=95.0)
mus = np.linspace(95, 125, 40)
collaterals = []
from pdamm.amm.gaussian_mixture import compute_collateral
for target_mu in mus:
    c = compute_collateral(
        amm2.params,
        GaussianMixtureParams.single(mu=target_mu, sigma=10.0),
        amm2.k,
    )
    collaterals.append(c)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(mus, collaterals)
ax.set_xlabel("Target mu")
ax.set_ylabel("Collateral required")
ax.set_title("Collateral vs. distribution shift size (sigma fixed at 10)")
plt.tight_layout()
plt.show()


## Opposing traders: bull vs. bear

In [ ]:
amm3 = make_amm(mu=100.0, b=100.0)
bull = InformedTrader("bull", GaussianMixtureParams.single(mu=120.0, sigma=8.0), step_fraction=0.2)
bear = InformedTrader("bear", GaussianMixtureParams.single(mu=80.0, sigma=8.0), step_fraction=0.2)

hist_bull, hist_bear, hist_amm = [], [], []
for _ in range(80):
    bull.step(amm3)
    bear.step(amm3)
    hist_amm.append(amm3.current_mu)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hist_amm, label="AMM mu")
ax.axhline(120.0, color="green", linestyle="--", label="Bull target")
ax.axhline(80.0, color="red", linestyle="--", label="Bear target")
ax.axhline(100.0, color="gray", linestyle=":", label="Initial")
ax.set_title("Opposing traders — market stays near equilibrium")
ax.legend()
plt.tight_layout()
plt.show()
